In [1]:
# ============================================================
# PHASE 1 - Data Loading + Data Understanding
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
BASE_DIR   = r"C:\Users\yipch\ecommerce-analytics-portfolio"
RAW_DIR    = os.path.join(BASE_DIR, "data", "raw")
SAMPLE_DIR = os.path.join(BASE_DIR, "data", "samples")

OCT_FILE   = os.path.join(RAW_DIR, "2019-Oct.csv")
NOV_FILE   = os.path.join(RAW_DIR, "2019-Nov.csv")

print("✅ Setup complete. Paths configured.")
print(f"   Oct file exists: {os.path.exists(OCT_FILE)}")
print(f"   Nov file exists: {os.path.exists(NOV_FILE)}")

✅ Setup complete. Paths configured.
   Oct file exists: True
   Nov file exists: True


In [2]:
# ============================================================
# STEP 3 - Peek without loading full file
# ============================================================

def peek_file(filepath, n_rows=5):
    """Read only first N rows to understand structure."""
    filename = os.path.basename(filepath)
    size_mb  = os.path.getsize(filepath) / (1024 * 1024)

    print(f"\n{'='*55}")
    print(f"  FILE : {filename}  ({size_mb:.1f} MB)")
    print(f"{'='*55}")

    df = pd.read_csv(filepath, nrows=n_rows)

    print(f"\n📐 Shape Preview  : {n_rows} rows × {df.shape[1]} columns")
    print(f"\n📋 Column Names   :")
    for i, col in enumerate(df.columns):
        print(f"   [{i}] {col} — dtype: {df[col].dtype}")

    print(f"\n👀 First {n_rows} Rows :")
    return df

oct_peek = peek_file(OCT_FILE)
display(oct_peek)


  FILE : 2019-Oct.csv  (5406.0 MB)

📐 Shape Preview  : 5 rows × 9 columns

📋 Column Names   :
   [0] event_time — dtype: str
   [1] event_type — dtype: str
   [2] product_id — dtype: int64
   [3] category_id — dtype: int64
   [4] category_code — dtype: str
   [5] brand — dtype: str
   [6] price — dtype: float64
   [7] user_id — dtype: int64
   [8] user_session — dtype: str

👀 First 5 Rows :


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [3]:
# ============================================================
# STEP 4 - Count rows without loading full data
# ============================================================

def count_rows(filepath):
    """Count rows by reading in chunks — memory safe."""
    filename = os.path.basename(filepath)
    count = 0
    for chunk in pd.read_csv(filepath, chunksize=100_000):
        count += len(chunk)
    print(f"  📊 {filename}: {count:,} total rows")
    return count

print("Counting rows (this takes ~30 seconds per file)...\n")
oct_rows = count_rows(OCT_FILE)
nov_rows = count_rows(NOV_FILE)
total    = oct_rows + nov_rows
print(f"\n  📦 TOTAL across both files: {total:,} rows")

Counting rows (this takes ~30 seconds per file)...

  📊 2019-Oct.csv: 42,448,764 total rows
  📊 2019-Nov.csv: 67,501,979 total rows

  📦 TOTAL across both files: 109,950,743 rows


In [6]:
# ============================================================
# STEP 5 - Professional User-Based Sampling (FIXED ARCHITECTURE)
# ============================================================
# Why user-based?
# → Preserves each user's full behavior chain (view→cart→purchase)
# → Valid for funnel, cohort, RFM, retention, CLV
# → Avoids broken sessions that distort conversion rates
# ============================================================

import pandas as pd
import numpy as np
import os

SAMPLE_USERS = 50_000    # sample N unique users per file
RANDOM_STATE = 42

def user_based_sample(filepath, n_users=SAMPLE_USERS, random_state=RANDOM_STATE):
    filename = os.path.basename(filepath)
    print(f"\n{'='*60}")
    print(f"  Loading: {filename}")
    print(f"{'='*60}")

    # --- Load full file in chunks ---
    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=200_000,
                             parse_dates=["event_time"]):
        chunks.append(chunk)
    df_full = pd.concat(chunks, ignore_index=True)
    print(f"\n  Full shape : {df_full.shape[0]:,} rows × {df_full.shape[1]} cols")
    print(f"  Unique users (full) : {df_full['user_id'].nunique():,}")

    # ─────────────────────────────────────────
    # DATASET 1: User-based sample (main)
    # Keep ALL events for sampled users
    # ─────────────────────────────────────────
    all_users    = df_full["user_id"].unique()
    sampled_users = np.random.RandomState(random_state).choice(
        all_users,
        size=min(n_users, len(all_users)),
        replace=False
    )
    df_main = df_full[df_full["user_id"].isin(sampled_users)].copy()

    print(f"\n  [DATASET 1] User-based sample:")
    print(f"    Users sampled  : {df_main['user_id'].nunique():,}")
    print(f"    Rows kept      : {df_main.shape[0]:,}")
    print(f"    Avg events/user: {df_main.shape[0]/df_main['user_id'].nunique():.1f}")

    # ─────────────────────────────────────────
    # DATASET 2: Sanity check — event distribution
    # Confirm sample is not biased
    # ─────────────────────────────────────────
    print(f"\n  [DATASET 2] Event distribution sanity check:")
    print(f"    {'event_type':<12} | {'FULL':>8} | {'SAMPLE':>8} | {'BIAS?':>6}")
    print(f"    {'-'*46}")

    full_dist   = df_full["event_type"].value_counts(normalize=True)
    sample_dist = df_main["event_type"].value_counts(normalize=True)

    bias_ok = True
    for evt in full_dist.index:
        full_pct   = full_dist.get(evt, 0)
        sample_pct = sample_dist.get(evt, 0)
        diff       = abs(full_pct - sample_pct)
        flag       = "✅" if diff < 0.02 else "⚠️ BIAS"
        if diff >= 0.02:
            bias_ok = False
        print(f"    {evt:<12} | {full_pct:>7.1%} | {sample_pct:>7.1%} | {flag}")

    if bias_ok:
        print(f"\n    ✅ No significant bias detected (diff < 2%)")
    else:
        print(f"\n    ⚠️  Bias detected — consider increasing SAMPLE_USERS")

    # ─────────────────────────────────────────
    # DATASET 3: Purchase segmentation labels
    # Label each user: purchased=1, non-purchased=0
    # Used for: conversion analysis + ML (bonus)
    # ─────────────────────────────────────────
    purchasers     = set(df_main[df_main["event_type"] == "purchase"]["user_id"])
    non_purchasers = set(df_main[df_main["event_type"] != "purchase"]["user_id"]) - purchasers

    df_user_labels = pd.DataFrame({
        "user_id"   : list(purchasers) + list(non_purchasers),
        "purchased" : [1] * len(purchasers) + [0] * len(non_purchasers)
    })

    print(f"\n  [DATASET 3] Purchase segmentation labels:")
    print(f"    Purchased (1)     : {len(purchasers):,}  users")
    print(f"    Non-purchased (0) : {len(non_purchasers):,}  users")
    print(f"    Conversion rate   : {len(purchasers)/(len(purchasers)+len(non_purchasers)):.2%}")

    return df_main, df_user_labels, df_full

# ─────────────────────────────────────────
# Run for both months
# ─────────────────────────────────────────
df_oct_main, df_oct_labels, df_oct_full = user_based_sample(OCT_FILE)
df_nov_main, df_nov_labels, df_nov_full = user_based_sample(NOV_FILE)


  Loading: 2019-Oct.csv

  Full shape : 42,448,764 rows × 9 cols
  Unique users (full) : 3,022,290

  [DATASET 1] User-based sample:
    Users sampled  : 50,000
    Rows kept      : 699,708
    Avg events/user: 14.0

  [DATASET 2] Event distribution sanity check:
    event_type   |     FULL |   SAMPLE |  BIAS?
    ----------------------------------------------
    view         |   96.1% |   96.2% | ✅
    cart         |    2.2% |    2.1% | ✅
    purchase     |    1.7% |    1.7% | ✅

    ✅ No significant bias detected (diff < 2%)

  [DATASET 3] Purchase segmentation labels:
    Purchased (1)     : 5,626  users
    Non-purchased (0) : 44,374  users
    Conversion rate   : 11.25%

  Loading: 2019-Nov.csv

  Full shape : 67,501,979 rows × 9 cols
  Unique users (full) : 3,696,117

  [DATASET 1] User-based sample:
    Users sampled  : 50,000
    Rows kept      : 910,694
    Avg events/user: 18.2

  [DATASET 2] Event distribution sanity check:
    event_type   |     FULL |   SAMPLE |  BIAS?
 

In [7]:
# ============================================================
# STEP 6 - Combine + Tag month
# ============================================================

df_oct_main["month"] = "2019-Oct"
df_nov_main["month"] = "2019-Nov"

df_oct_labels["month"] = "2019-Oct"
df_nov_labels["month"] = "2019-Nov"

# Main analysis dataset
df = pd.concat([df_oct_main, df_nov_main], ignore_index=True)

# Purchase segmentation dataset
df_labels = pd.concat([df_oct_labels, df_nov_labels], ignore_index=True)
# A user in both months → keep as purchased=1 if purchased in either
df_labels = (df_labels
             .groupby("user_id")
             .agg(purchased=("purchased","max"), month=("month","last"))
             .reset_index())

print("✅ MAIN dataset (user-based sample):")
print(f"   Shape  : {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"   Users  : {df['user_id'].nunique():,}")
print(f"   Months : {df['month'].value_counts().to_dict()}")

print("\n✅ LABELS dataset (purchase segmentation):")
print(f"   Shape  : {df_labels.shape[0]:,} rows")
print(df_labels['purchased'].value_counts().rename({1:'Purchased',0:'Not Purchased'}).to_string())

✅ MAIN dataset (user-based sample):
   Shape  : 1,610,402 rows × 10 cols
   Users  : 99,693
   Months : {'2019-Nov': 910694, '2019-Oct': 699708}

✅ LABELS dataset (purchase segmentation):
   Shape  : 99,693 rows
purchased
Not Purchased    87995
Purchased        11698


In [8]:
# ============================================================
# STEP 7 - Schema Glossary (what each column means)
# ============================================================

schema = {
    "event_time"    : "Timestamp of the event (UTC)",
    "event_type"    : "Type of action: view / cart / purchase",
    "product_id"    : "Unique ID for each product",
    "category_id"   : "Numeric ID for product category",
    "category_code" : "Human-readable category (e.g. electronics.smartphone)",
    "brand"         : "Brand name of the product",
    "price"         : "Product price in USD",
    "user_id"       : "Unique ID for each user",
    "user_session"  : "Session ID — groups events in one browsing session",
    "month"         : "Added by us — which month the event belongs to",
}

print("📖 SCHEMA GLOSSARY")
print("=" * 55)
for col, description in schema.items():
    present = "✅" if col in df.columns else "❌ NOT FOUND"
    print(f"  {present}  {col:<16} → {description}")

📖 SCHEMA GLOSSARY
  ✅  event_time       → Timestamp of the event (UTC)
  ✅  event_type       → Type of action: view / cart / purchase
  ✅  product_id       → Unique ID for each product
  ✅  category_id      → Numeric ID for product category
  ✅  category_code    → Human-readable category (e.g. electronics.smartphone)
  ✅  brand            → Brand name of the product
  ✅  price            → Product price in USD
  ✅  user_id          → Unique ID for each user
  ✅  user_session     → Session ID — groups events in one browsing session
  ✅  month            → Added by us — which month the event belongs to


In [9]:
# ============================================================
# STEP 8 - Data Quality Report
# ============================================================

print("=" * 60)
print("  DATA QUALITY REPORT")
print("=" * 60)

# --- 1. Basic shape ---
print(f"\n📐 Shape : {df.shape[0]:,} rows  ×  {df.shape[1]} columns")

# --- 2. Data types ---
print(f"\n📋 Data Types :")
print(df.dtypes.to_string())

# --- 3. Missing values ---
print(f"\n❓ Missing Values :")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    "Missing Count" : missing,
    "Missing %"     : missing_pct
})
print(missing_report[missing_report["Missing Count"] > 0].to_string())
if missing_report["Missing Count"].sum() == 0:
    print("   No missing values found!")

# --- 4. Duplicates ---
print(f"\n🔁 Duplicate Rows : {df.duplicated().sum():,}")

# --- 5. Event type breakdown ---
print(f"\n🎯 Event Type Breakdown :")
evt = df["event_type"].value_counts()
evt_pct = (evt / len(df) * 100).round(2)
for e in evt.index:
    print(f"   {e:<12} : {evt[e]:>8,}  ({evt_pct[e]:.2f}%)")

# --- 6. Date range ---
print(f"\n📅 Date Range :")
print(f"   Earliest : {df['event_time'].min()}")
print(f"   Latest   : {df['event_time'].max()}")

# --- 7. Unique counts ---
print(f"\n🔢 Unique Value Counts :")
unique_cols = ["user_id", "product_id", "category_id",
               "category_code", "brand", "user_session"]
for col in unique_cols:
    print(f"   {col:<20} : {df[col].nunique():>8,} unique values")

# --- 8. Price stats ---
print(f"\n💰 Price Statistics :")
print(df["price"].describe().round(2).to_string())

# --- 9. Zero / negative prices ---
zero_price = (df["price"] == 0).sum()
neg_price  = (df["price"] < 0).sum()
print(f"\n   ⚠️  Zero prices    : {zero_price:,}")
print(f"   ⚠️  Negative prices: {neg_price:,}")

print("\n" + "=" * 60)
print("  ✅ Quality Report Complete")
print("=" * 60)

  DATA QUALITY REPORT

📐 Shape : 1,610,402 rows  ×  10 columns

📋 Data Types :
event_time       datetime64[us, UTC]
event_type                       str
product_id                     int64
category_id                    int64
category_code                    str
brand                            str
price                        float64
user_id                        int64
user_session                     str
month                            str

❓ Missing Values :
               Missing Count  Missing %
category_code         517940      32.16
brand                 222747      13.83

🔁 Duplicate Rows : 1,709

🎯 Event Type Breakdown :
   view         : 1,528,765  (94.93%)
   cart         :   57,034  (3.54%)
   purchase     :   24,603  (1.53%)

📅 Date Range :
   Earliest : 2019-10-01 00:01:39+00:00
   Latest   : 2019-11-30 23:59:38+00:00

🔢 Unique Value Counts :
   user_id              :   99,693 unique values
   product_id           :   96,282 unique values
   category_id          :     

In [10]:
# ============================================================
# STEP 9 - Save all 3 datasets
# ============================================================

# Dataset 1: Main (user-based, for all analysis)
path_main = os.path.join(SAMPLE_DIR, "sampled_main.csv")
df.to_csv(path_main, index=False)

# Dataset 2: Labels (for conversion + ML)
path_labels = os.path.join(SAMPLE_DIR, "sampled_user_labels.csv")
df_labels.to_csv(path_labels, index=False)

for path, label in [(path_main, "Main dataset"),
                    (path_labels, "Labels dataset")]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    rows    = pd.read_csv(path).shape[0]
    print(f"✅ {label}")
    print(f"   Path : {path}")
    print(f"   Size : {size_mb:.1f} MB  |  Rows: {rows:,}\n")

✅ Main dataset
   Path : C:\Users\yipch\ecommerce-analytics-portfolio\data\samples\sampled_main.csv
   Size : 223.3 MB  |  Rows: 1,610,402

✅ Labels dataset
   Path : C:\Users\yipch\ecommerce-analytics-portfolio\data\samples\sampled_user_labels.csv
   Size : 2.1 MB  |  Rows: 99,693

